# Tamil Nadu Temples Category Prediction Pipeline
This notebook demonstrates how to load, clean, pre-process, and train a boosted machine learning model (LightGBM) to predict the **12A Income Category** of temples in Tamil Nadu based on categorical features and temple names.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import lightgbm as lgb
import joblib

## 1. Load the Dataset

In [ ]:
data_path = r"C:\Users\adity\OneDrive\Desktop\tn_temples_consolidated.xlsx"
print(f"Loading dataset from {data_path}...")
df = pd.read_excel(data_path)
print(f"Shape: {df.shape}")
df.head()

## 2. Preprocessing & Feature Engineering
- Handle missing values.
- Extract `pincode_prefix` (first 3 digits of pincode) as a regional geographical feature.
- Encode target labels.

In [ ]:
# Drop rows with missing target variable
df = df.dropna(subset=['temple_12a_category'])

# Handle missing values in other features
df['district'] = df['district'].fillna('Unknown')
df['temple_type'] = df['temple_type'].fillna('Unknown')
df['temple_listing_type'] = df['temple_listing_type'].fillna('Unknown')
df['temple_name'] = df['temple_name'].fillna('')

# Feature Engineering: Clean pincode and extract first 3 digits
df['pincode_str'] = df['pincode'].fillna('000000').astype(str).str.replace(r'\.0$', '', regex=True)
df['pincode_prefix'] = df['pincode_str'].str[:3]
df['pincode_prefix'] = df['pincode_prefix'].apply(lambda x: x if x.isdigit() and len(x) == 3 else '000')

# Target definition & encoding
target_col = 'temple_12a_category'
X = df[['temple_type', 'temple_listing_type', 'district', 'pincode_prefix', 'temple_name']]
y = df[target_col]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 4. Pipeline Setup & LightGBM Training

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['temple_type', 'temple_listing_type', 'district', 'pincode_prefix']),
        ('text', TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2)), 'temple_name')
    ],
    remainder='drop'
)

print("Preprocessing and transforming data...")
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Training LightGBM Classifier...")
model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model.fit(X_train_transformed, y_train)
print("Training finished!")

## 5. Model Evaluation

In [ ]:
y_pred = model.predict(X_test_transformed)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 6. Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.tight_layout()
plt.show()

## 7. Feature Importance Plot (Top 20)

In [ ]:
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(['temple_type', 'temple_listing_type', 'district', 'pincode_prefix'])
text_names = preprocessor.named_transformers_['text'].get_feature_names_out()
feature_names = np.concatenate([cat_names, text_names])

importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 8))
sns.barplot(x=importances[indices[:20]], y=feature_names[indices[:20]], hue=feature_names[indices[:20]], palette='viridis', legend=False)
plt.title('Top 20 Most Important Features')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

## 8. Save the Pipeline

In [ ]:
save_data = {
    'preprocessor': preprocessor,
    'model': model,
    'label_encoder': le
}
save_path = 'temple_boosted_model.joblib'
joblib.dump(save_data, save_path)
print(f"Saved pipeline to {save_path}")